THE DATASETS GENERATED IN THE PREPROCESSING STEP ARE PROVIDED BY THE FOLLOWING PUBLIC LINKS

Install the following modules if necessary

In [25]:
tuple_classical_country_electronic_hiphop : tuple[str,str] = "https://drive.google.com/file/d/1bAlexFnXYDtT8wDOwCqPUWsHu5n2DzPk/view?usp=share_link","ClassicalCountryHiphopElectronic"
tuple_electronic_hiphop_jazz_rock: tuple[str,str] = "https://drive.google.com/file/d/1WwHkyXTXc9e3BPG1u7sb5Vbf6grHxg6l/view?usp=share_link", "ElectronicHipHopJazzRock"

data_structures : list[tuple] = [tuple_classical_country_electronic_hiphop,tuple_electronic_hiphop_jazz_rock]


In [ ]:
#!pip install datasets evaluate scikit-learn matplotlib 

Modules used for this project, the datasets are available at the mounted drive

In [8]:
from datasets import load_dataset,ClassLabel, Dataset, DatasetDict
from transformers import Trainer,AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,EarlyStoppingCallback
import numpy as np
from evaluate import load, EvaluationModule
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import gdown
import os

We will use BERT with a classifier head for this project

In [5]:
#used model
model_name : str = "bert-base-uncased"
#download model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1438.38it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

Tokenize the lyrics, if too long they will be truncated to the max length of 512. If too short, padding will be applied

In [6]:
#initialise the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_function (set : Dataset) :
    return tokenizer(set['lyrics'], padding="max_length", truncation=True,max_length=512)

Convert the dataset into a format that BERT can work with

In [7]:
def convertDatasetForBERT(dataset : DatasetDict) -> Dataset: 
    genres : list = list(set(dataset["train"]["genre"]))
    #set classlabel
    class_label : ClassLabel = ClassLabel(names=genres)
    #set the classlabel
    dataset : Dataset = dataset["train"].cast_column("genre", class_label)
    #create split
    dataset_split : Dataset = dataset.train_test_split(test_size=0.2, seed=69, stratify_by_column="genre")
    #tokenize the dataset
    tokenized_dataset : Dataset = dataset_split.map(tokenize_function, batched=True)
    #rename classlabel
    tokenized_dataset = tokenized_dataset.rename_column("genre", "labels")
    return tokenized_dataset

A custom metric so we can understand the training progress

In [9]:
#load metrics
metric_f1 : EvaluationModule = load("f1")
metric_acc : EvaluationModule = load("accuracy")
#custom metric to evaluate on
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": metric_acc.compute(
            predictions=preds,
            references=labels
        )["accuracy"],

        "f1_macro": metric_f1.compute(
            predictions=preds,
            references=labels,
            average="macro"  
        )["f1"],
    }

Arguments for the HuggingFace Trainer

In [10]:
#set the traning arguments
#label smoothing, low learning rate, after epoch 3 overfitting becomes a problem so we stop after that
training_args : TrainingArguments = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    label_smoothing_factor=0.05,
    fp16=True,
)

In [30]:
def download_datasets(data_structures : tuple):
    basepath : str = "datasets"
    for structure in data_structures:
        os.makedirs(f"{basepath}", exist_ok=True)
        gdown.download(structure[0], f"{basepath}/{structure[1]}.csv", fuzzy=True)

In [31]:
download_datasets(data_structures=data_structures)

Downloading...
From: https://drive.google.com/uc?id=1bAlexFnXYDtT8wDOwCqPUWsHu5n2DzPk
To: /Users/saunagang/nlp/datasets/ClassicalCountryHiphopElectronic.csv
100%|██████████| 15.9M/15.9M [00:00<00:00, 18.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WwHkyXTXc9e3BPG1u7sb5Vbf6grHxg6l
To: /Users/saunagang/nlp/datasets/ElectronicHipHopJazzRock.csv
100%|██████████| 14.7M/14.7M [00:00<00:00, 15.0MB/s]


In [ ]:
#load dataset
dataset = load_dataset("csv", data_files="/content/drive/MyDrive/lyricsclassificationdata/archive/balancedSubset.csv")

In [ ]:
#tokenise the dataset
tokenized_dataset = convertDatasetForBERT(dataset=dataset)

In [ ]:
#initialise the trainer, early stopping to prevent overfitting
trainer = Trainer(
    model=model,                        
    args=training_args,                 
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],   
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()
trainer.save_model("models/model10kClassicalCountryHipHopElectronic")

Evaluation & Confusion Matrix

In [ ]:
#evaluate
predictions = trainer.predict(tokenized_dataset['test'])

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=-1)

cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.show()

print(cm)

MODEL PIPELINE
